In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, FloatType, StringType
)
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml import Pipeline

print("Libraries imported successfully.")
print(f"Spark version: {spark.version}")

Libraries imported successfully.
Spark version: 4.2.0


In [0]:
SILVER_TABLE = "healthcare_readmission.readmission.silver_patient_data"

silver_df = spark.table(SILVER_TABLE)

print(f"Silver table loaded: {SILVER_TABLE}")
print(f"Rows: {silver_df.count():,}")
print(f"Columns: {len(silver_df.columns)}")

Silver table loaded: healthcare_readmission.readmission.silver_patient_data
Rows: 101,766
Columns: 48


In [0]:
print("=== SILVER TABLE SCHEMA ===")
silver_df.printSchema()

row_count = silver_df.count()
col_count = len(silver_df.columns)
print(f"\nTotal rows:    {row_count:,}")
print(f"Total columns: {col_count}")

print("\n=== TARGET DISTRIBUTION: readmitted ===")
target_dist = (
    silver_df.groupBy("readmitted")
    .count()
    .withColumn("percentage", F.round((F.col("count") / F.lit(row_count)) * 100, 2))
    .orderBy(F.col("count").desc())
)
display(target_dist)

print("\nText summary:")
for row in target_dist.collect():
    print(f"  {row['readmitted']:<5s}  count: {row['count']:>7,}  ({row['percentage']}%)")

=== SILVER TABLE SCHEMA ===
root
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- number_outpatient: integer (nullable = true)
 |-- number_emergency: integer (nullable = true)
 |-- number_inpatient: integer (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: integer (nullable = true)
 |-- max_glu_serum: string (nullable = true)
 |-- A1Cresult: st

readmitted,count,percentage
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16



Text summary:
  NO     count:  54,864  (53.91%)
  >30    count:  35,545  (34.93%)
  <30    count:  11,357  (11.16%)


In [0]:
string_cols = [
    f.name for f in silver_df.schema.fields
    if isinstance(f.dataType, StringType)
]

integer_cols = [
    f.name for f in silver_df.schema.fields
    if isinstance(f.dataType, IntegerType)
]

print("=== STRING (CATEGORICAL) COLUMNS ===")
for c in string_cols:
    print(f"  {c}")
print(f"  Total: {len(string_cols)}")

print("\n=== INTEGER COLUMNS ===")
for c in integer_cols:
    print(f"  {c}")
print(f"  Total: {len(integer_cols)}")

print(f"\nTotal columns: {len(string_cols) + len(integer_cols)}")

=== STRING (CATEGORICAL) COLUMNS ===
  race
  gender
  weight
  payer_code
  medical_specialty
  diag_1
  diag_2
  diag_3
  max_glu_serum
  A1Cresult
  metformin
  repaglinide
  nateglinide
  chlorpropamide
  glimepiride
  acetohexamide
  glipizide
  glyburide
  tolbutamide
  pioglitazone
  rosiglitazone
  acarbose
  miglitol
  troglitazone
  tolazamide
  examide
  citoglipton
  insulin
  glyburide-metformin
  glipizide-metformin
  glimepiride-pioglitazone
  metformin-rosiglitazone
  metformin-pioglitazone
  change
  diabetesMed
  readmitted
  Total: 36

=== INTEGER COLUMNS ===
  age
  admission_type_id
  discharge_disposition_id
  admission_source_id
  time_in_hospital
  num_lab_procedures
  num_procedures
  num_medications
  number_outpatient
  number_emergency
  number_inpatient
  number_diagnoses
  Total: 12

Total columns: 48


In [0]:
target_col = "readmitted"

zero_variance_cols = ["examide", "citoglipton"]

print("=== ZERO-VARIANCE COLUMN VERIFICATION ===")
for c in zero_variance_cols:
    distinct_count = silver_df.select(c).distinct().count()
    print(f"  {c}: {distinct_count} distinct value(s)")

id_code_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

print(f"\n=== EXCLUSION SUMMARY ===")
print(f"  Target (excluded from features):      {target_col}")
print(f"  Zero-variance (no information):        {zero_variance_cols}")
print(f"  Integer-coded categoricals (to OHE):   {id_code_cols}")
print(f"\nAll other columns are retained as features.")
print(f"No columns are removed arbitrarily — each exclusion is justified.")

print("\n=== DATA LEAKAGE CHECK ===")
print("Identifiers (encounter_id, patient_nbr) were already removed in Silver.")
print("All remaining features are recorded DURING the hospital stay,")
print("before discharge — none represent post-discharge or post-readmission")
print("information. This prevents target leakage.")
print("\nNote: discharge_disposition_id is known at discharge time, so it is")
print("a legitimate pre-readmission feature, not a leakage source.")

=== ZERO-VARIANCE COLUMN VERIFICATION ===
  examide: 1 distinct value(s)
  citoglipton: 1 distinct value(s)

=== EXCLUSION SUMMARY ===
  Target (excluded from features):      readmitted
  Zero-variance (no information):        ['examide', 'citoglipton']
  Integer-coded categoricals (to OHE):   ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']

All other columns are retained as features.
No columns are removed arbitrarily — each exclusion is justified.

=== DATA LEAKAGE CHECK ===
Identifiers (encounter_id, patient_nbr) were already removed in Silver.
All remaining features are recorded DURING the hospital stay,
before discharge — none represent post-discharge or post-readmission
information. This prevents target leakage.

Note: discharge_disposition_id is known at discharge time, so it is
a legitimate pre-readmission feature, not a leakage source.


In [0]:
print("=== MISSING VALUE CHECK ===")
any_missing = False

for c in silver_df.columns:
    null_count = silver_df.filter(F.col(c).isNull()).count()
    if null_count > 0:
        print(f"  {c}: {null_count} NULLs")
        any_missing = True

if not any_missing:
    print("  No NULL values found in any column.")

qm_found = False
for c in silver_df.columns:
    field = [f for f in silver_df.schema.fields if f.name == c][0]
    if isinstance(field.dataType, StringType):
        cnt = silver_df.filter(F.col(c) == "?").count()
        if cnt > 0:
            print(f"  WARNING: '{c}' still has {cnt} '?' values!")
            qm_found = True

if not qm_found:
    print("  No '?' placeholder values remain.")

print("\nMissing value check complete — data is clean for encoding.")

=== MISSING VALUE CHECK ===
  No NULL values found in any column.
  No '?' placeholder values remain.

Missing value check complete — data is clean for encoding.


In [0]:
for c in id_code_cols:
    silver_df = silver_df.withColumn(c, F.col(c).cast("string"))

print("Converted categorical ID columns to string:")
for c in id_code_cols:
    print(f"  {c}")

categorical_features = [
    f.name for f in silver_df.schema.fields
    if isinstance(f.dataType, StringType)
    and f.name not in [target_col] + zero_variance_cols
]

numerical_features = [
    f.name for f in silver_df.schema.fields
    if isinstance(f.dataType, IntegerType)
]

print(f"\n=== FINAL FEATURE LISTS ===")
print(f"\nCategorical features ({len(categorical_features)}):")
for c in categorical_features:
    print(f"  {c}")

print(f"\nNumerical features ({len(numerical_features)}):")
for c in numerical_features:
    print(f"  {c}")

print(f"\nTotal features: {len(categorical_features) + len(numerical_features)}")
print(f"Excluded: target='{target_col}', zero-variance={zero_variance_cols}")

Converted categorical ID columns to string:
  admission_type_id
  discharge_disposition_id
  admission_source_id

=== FINAL FEATURE LISTS ===

Categorical features (36):
  race
  gender
  weight
  admission_type_id
  discharge_disposition_id
  admission_source_id
  payer_code
  medical_specialty
  diag_1
  diag_2
  diag_3
  max_glu_serum
  A1Cresult
  metformin
  repaglinide
  nateglinide
  chlorpropamide
  glimepiride
  acetohexamide
  glipizide
  glyburide
  tolbutamide
  pioglitazone
  rosiglitazone
  acarbose
  miglitol
  troglitazone
  tolazamide
  insulin
  glyburide-metformin
  glipizide-metformin
  glimepiride-pioglitazone
  metformin-rosiglitazone
  metformin-pioglitazone
  change
  diabetesMed

Numerical features (9):
  age
  time_in_hospital
  num_lab_procedures
  num_procedures
  num_medications
  number_outpatient
  number_emergency
  number_inpatient
  number_diagnoses

Total features: 45
Excluded: target='readmitted', zero-variance=['examide', 'citoglipton']


In [0]:
target_indexer = StringIndexer(
    inputCol="readmitted",
    outputCol="label",
    handleInvalid="keep"
)

target_model = target_indexer.fit(silver_df)

print("=== TARGET LABEL MAPPING ===")
for idx, label in enumerate(target_model.labels):
    count = silver_df.filter(F.col("readmitted") == label).count()
    print(f"  '{label}' -> label {idx}  ({count:,} records)")

print(f"\nThree-class classification problem preserved (no binarisation).")

=== TARGET LABEL MAPPING ===
  'NO' -> label 0  (54,864 records)
  '>30' -> label 1  (35,545 records)
  '<30' -> label 2  (11,357 records)

Three-class classification problem preserved (no binarisation).


In [0]:
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_features
]

# Single OneHotEncoder for ALL indexed categorical columns
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_features],
    outputCols=[f"{c}_ohe" for c in categorical_features],
    handleInvalid="keep"
)

print(f"Built {len(indexers)} StringIndexers for categorical features.")
print(f"Built 1 OneHotEncoder for {len(categorical_features)} indexed columns.")
print(f"\nEncoding flow (per categorical column):")
print(f"  string column -> {{col}}_idx  (StringIndexer)")
print(f"  {{col}}_idx     -> {{col}}_ohe  (OneHotEncoder)")

Built 36 StringIndexers for categorical features.
Built 1 OneHotEncoder for 36 indexed columns.

Encoding flow (per categorical column):
  string column -> {col}_idx  (StringIndexer)
  {col}_idx     -> {col}_ohe  (OneHotEncoder)


In [0]:
# ============================================================
# Cell 11: Build Numerical Feature Stages
# ============================================================
# Numerical features are already clean integers (no missing values
# — verified in Cell 7). We:
#   1. Assemble them into a single dense vector column
#   2. Apply StandardScaler (zero mean, unit variance)
#
# StandardScaler helps distance-based and gradient-based algorithms
# (logistic regression, SVM, neural networks) converge faster and
# treat all features equally. Tree-based models are scale-invariant
# but scaling does not harm them.
#
# withMean=True centres the data; since we have only 9 numerical
# features, the dense vector is small and memory-efficient.

# Assemble all numerical features into one vector
num_assembler = VectorAssembler(
    inputCols=numerical_features,
    outputCol="numerical_features"
)

# Scale the assembled numerical vector
num_scaler = StandardScaler(
    inputCol="numerical_features",
    outputCol="numerical_scaled",
    withMean=True,
    withStd=True
)

print(f"Built VectorAssembler for {len(numerical_features)} numerical features:")
for c in numerical_features:
    print(f"  {c}")
print(f"\nBuilt StandardScaler (withMean=True, withStd=True).")

Built VectorAssembler for 9 numerical features:
  age
  time_in_hospital
  num_lab_procedures
  num_procedures
  num_medications
  number_outpatient
  number_emergency
  number_inpatient
  number_diagnoses

Built StandardScaler (withMean=True, withStd=True).


In [0]:
# ============================================================
# Cell 12: Build Final VectorAssembler and Pipeline
# ============================================================
# The final VectorAssembler combines:
#   - All one-hot encoded categorical vectors ({col}_ohe)
#   - The scaled numerical vector (numerical_scaled)
# into a single 'features' column that ML models can consume.
#
# The Pipeline chains all stages in order so the entire
# transformation is reproducible and can be re-applied to new data.

# Collect all encoded categorical output column names
encoded_categorical_cols = [f"{c}_ohe" for c in categorical_features]

# Final assembler: encoded categoricals + scaled numericals
final_assembler = VectorAssembler(
    inputCols=encoded_categorical_cols + ["numerical_scaled"],
    outputCol="features"
)

# Build the complete pipeline
# Order matters: indexers -> encoder -> target_indexer ->
#   num_assembler -> num_scaler -> final_assembler
pipeline_stages = (
    indexers
    + [encoder, target_indexer, num_assembler, num_scaler, final_assembler]
)

feature_pipeline = Pipeline(stages=pipeline_stages)

print("=== PIPELINE STAGES ===")
for i, stage in enumerate(pipeline_stages):
    print(f"  Stage {i + 1}: {stage.__class__.__name__}")

print(f"\nTotal stages: {len(pipeline_stages)}")
print(f"Final output column: 'features' (assembled vector)")
print(f"Target output column: 'label' (numeric, 3 classes)")

=== PIPELINE STAGES ===
  Stage 1: StringIndexer
  Stage 2: StringIndexer
  Stage 3: StringIndexer
  Stage 4: StringIndexer
  Stage 5: StringIndexer
  Stage 6: StringIndexer
  Stage 7: StringIndexer
  Stage 8: StringIndexer
  Stage 9: StringIndexer
  Stage 10: StringIndexer
  Stage 11: StringIndexer
  Stage 12: StringIndexer
  Stage 13: StringIndexer
  Stage 14: StringIndexer
  Stage 15: StringIndexer
  Stage 16: StringIndexer
  Stage 17: StringIndexer
  Stage 18: StringIndexer
  Stage 19: StringIndexer
  Stage 20: StringIndexer
  Stage 21: StringIndexer
  Stage 22: StringIndexer
  Stage 23: StringIndexer
  Stage 24: StringIndexer
  Stage 25: StringIndexer
  Stage 26: StringIndexer
  Stage 27: StringIndexer
  Stage 28: StringIndexer
  Stage 29: StringIndexer
  Stage 30: StringIndexer
  Stage 31: StringIndexer
  Stage 32: StringIndexer
  Stage 33: StringIndexer
  Stage 34: StringIndexer
  Stage 35: StringIndexer
  Stage 36: StringIndexer
  Stage 37: OneHotEncoder
  Stage 38: StringIndex

In [0]:
# ============================================================
# Cell 13: Fit Pipeline and Transform to Create Gold Dataset
# ============================================================
# Fit the pipeline on the Silver data (learns index mappings,
# scaling statistics), then transform to produce the ML-ready
# Gold dataset.
#
# The resulting DataFrame has intermediate columns from each stage.
# We select only what is needed for ML: 'features' and 'label',
# plus the original 'readmitted' string for interpretability.

print("Fitting feature engineering pipeline...")
fitted_pipeline = feature_pipeline.fit(silver_df)

print("Transforming Silver data to Gold...")
gold_df = fitted_pipeline.transform(silver_df)

# Keep only the ML-ready columns
gold_df = gold_df.select("features", "label", "readmitted")

print("\nGold dataset created successfully.")
print(f"Rows:    {gold_df.count():,}")
print(f"Columns: {gold_df.columns}")

Fitting feature engineering pipeline...
Transforming Silver data to Gold...

Gold dataset created successfully.
Rows:    101,766
Columns: ['features', 'label', 'readmitted']


In [0]:
# ============================================================
# Cell 14: Inspect the Gold Dataset
# ============================================================
# Verify the feature vector and label look correct before saving.

print("=== GOLD DATASET SCHEMA ===")
gold_df.printSchema()

print("\n=== SAMPLE RECORDS (first 5 rows) ===")
display(gold_df.limit(5))

# Show feature vector details from the first row
print("\n=== FEATURE VECTOR DETAILS (first row) ===")
sample_row = gold_df.first()
print(f"  Feature vector size:  {len(sample_row['features'])}")
print(f"  Feature vector type: {type(sample_row['features']).__name__}")
print(f"  Label value:          {sample_row['label']}")
print(f"  Original target:      {sample_row['readmitted']}")

# Show label-to-readmitted mapping
total_rows = gold_df.count()
print("\n=== LABEL DISTRIBUTION ===")
label_dist = (
    gold_df.groupBy("label", "readmitted")
    .count()
    .withColumn("percentage", F.round((F.col("count") / F.lit(total_rows)) * 100, 2))
    .orderBy("label")
)
display(label_dist)

=== GOLD DATASET SCHEMA ===
root
 |-- features: vectorudt (nullable = true)
 |-- label: double (nullable = false)
 |-- readmitted: string (nullable = true)


=== SAMPLE RECORDS (first 5 rows) ===


features,label,readmitted
"{""type"":""0"",""size"":""2543"",""indices"":[""1"",""7"",""11"",""22"",""32"",""58"",""76"",""95"",""169"",""952"",""1651"",""2428"",""2433"",""2438"",""2443"",""2448"",""2453"",""2458"",""2463"",""2466"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2509"",""2511"",""2516"",""2519"",""2522"",""2525"",""2529"",""2531"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""-1.31530241936465"",""-0.467650410164546"",""1.0625177454290071"",""-0.7853938799691554"",""-0.12572573444311633"",""-0.2914600533812482"",""-0.21261913890554268"",""0.28857762891627553"",""0.8157804701984116""]}",1.0,>30
"{""type"":""0"",""size"":""2543"",""indices"":[""0"",""7"",""11"",""23"",""31"",""59"",""76"",""100"",""354"",""958"",""1745"",""2428"",""2433"",""2439"",""2443"",""2448"",""2453"",""2458"",""2463"",""2467"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2508"",""2511"",""2516"",""2519"",""2522"",""2525"",""2529"",""2531"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""0.5666563946707227"",""3.2173086790405527"",""-0.7672747298808162"",""0.9733045171646266"",""1.473769079310224"",""-0.2914600533812482"",""-0.21261913890554268"",""-0.5032737201897166"",""0.29861046346916253""]}",0.0,NO
"{""type"":""0"",""size"":""2543"",""indices"":[""1"",""8"",""11"",""22"",""33"",""58"",""76"",""95"",""246"",""924"",""1677"",""2428"",""2433"",""2438"",""2443"",""2448"",""2453"",""2458"",""2463"",""2466"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2506"",""2511"",""2516"",""2519"",""2522"",""2525"",""2528"",""2532"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""-0.6879828146861925"",""-0.467650410164546"",""0.4525869203257326"",""-0.7853938799691554"",""-0.3718018596359379"",""-0.2914600533812482"",""-0.21261913890554268"",""1.0804289780222678"",""0.8157804701984116""]}",0.0,NO
"{""type"":""0"",""size"":""2543"",""indices"":[""1"",""7"",""11"",""24"",""31"",""63"",""76"",""95"",""219"",""929"",""1638"",""2428"",""2433"",""2438"",""2443"",""2448"",""2453"",""2458"",""2463"",""2466"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2507"",""2511"",""2516"",""2519"",""2522"",""2525"",""2528"",""2531"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""-0.6879828146861925"",""-1.1376429718382004"",""-0.462309317329179"",""-0.1991610809245614"",""-1.2330682978108136"",""-0.2914600533812482"",""-0.21261913890554268"",""-0.5032737201897166"",""-1.7700695634478336""]}",1.0,>30
"{""type"":""0"",""size"":""2543"",""indices"":[""0"",""8"",""11"",""24"",""31"",""63"",""76"",""95"",""414"",""897"",""1970"",""2428"",""243


=== FEATURE VECTOR DETAILS (first row) ===
  Feature vector size:  2543
  Feature vector type: SparseVector
  Label value:          1.0
  Original target:      >30

=== LABEL DISTRIBUTION ===


label,readmitted,count,percentage
0.0,NO,54864,53.91
1.0,>30,35545,34.93
2.0,<30,11357,11.16


In [0]:
# ============================================================
# Cell 15: Save Gold Dataset as Delta Table
# ============================================================
# Write the ML-ready Gold dataset to:
#   healthcare_readmission.readmission.gold_patient_features
# This table contains the feature vector and numeric label,
# ready for model training in a subsequent notebook.
# We do NOT modify the Silver table.

GOLD_TABLE = "healthcare_readmission.readmission.gold_patient_features"

print(f"Writing Gold Delta table: {GOLD_TABLE}")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(GOLD_TABLE)

print(f"\nGold table written successfully: {GOLD_TABLE}")

Writing Gold Delta table: healthcare_readmission.readmission.gold_patient_features

Gold table written successfully: healthcare_readmission.readmission.gold_patient_features


In [0]:
# ============================================================
# Cell 16: Verify the Gold Table
# ============================================================
# Read the saved Gold table back and verify:
#   - Row count matches Silver (101,766)
#   - Schema is correct (features vector + label + readmitted)
#   - Three-class distribution is preserved

# Read back from the saved Delta table
gold_verify_df = spark.table(GOLD_TABLE)

print("=== GOLD TABLE VERIFICATION ===")

# 1. Schema
print("\n1. Schema:")
gold_verify_df.printSchema()

# 2. Row count
gold_rows = gold_verify_df.count()
print(f"\n2. Row count: {gold_rows:,}")

# 3. Sample records
print(f"\n3. Sample records (first 5 rows):")
display(gold_verify_df.limit(5))

# 4. Class distribution
print(f"\n4. Class distribution:")
class_dist = (
    gold_verify_df.groupBy("label", "readmitted")
    .count()
    .withColumn("percentage", F.round((F.col("count") / F.lit(gold_rows)) * 100, 2))
    .orderBy("label")
)
display(class_dist)

print("\nText summary:")
for row in class_dist.collect():
    print(f"  label {row['label']}  ({row['readmitted']:<5s})"
          f"  count: {row['count']:>7,}  ({row['percentage']}%)")

# 5. Final verification summary
print(f"\n=== VERIFICATION SUMMARY ===")
row_match = gold_rows == 101766
class_match = class_dist.count() == 3
print(f"  Rows preserved: {gold_rows:,} (expected 101,766) {'✓' if row_match else '✗'}")
print(f"  Columns: {gold_verify_df.columns}")
print(f"  Three-class target preserved: {'✓' if class_match else '✗'}")
print(f"\nGold table '{GOLD_TABLE}' is ready for model training.")

=== GOLD TABLE VERIFICATION ===

1. Schema:
root
 |-- features: vectorudt (nullable = true)
 |-- label: double (nullable = true)
 |-- readmitted: string (nullable = true)


2. Row count: 101,766

3. Sample records (first 5 rows):


features,label,readmitted
"{""type"":""0"",""size"":""2543"",""indices"":[""1"",""7"",""11"",""22"",""32"",""58"",""76"",""95"",""169"",""952"",""1651"",""2428"",""2433"",""2438"",""2443"",""2448"",""2453"",""2458"",""2463"",""2466"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2509"",""2511"",""2516"",""2519"",""2522"",""2525"",""2529"",""2531"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""-1.31530241936465"",""-0.467650410164546"",""1.0625177454290071"",""-0.7853938799691554"",""-0.12572573444311633"",""-0.2914600533812482"",""-0.21261913890554268"",""0.28857762891627553"",""0.8157804701984116""]}",1.0,>30
"{""type"":""0"",""size"":""2543"",""indices"":[""0"",""7"",""11"",""23"",""31"",""59"",""76"",""100"",""354"",""958"",""1745"",""2428"",""2433"",""2439"",""2443"",""2448"",""2453"",""2458"",""2463"",""2467"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2508"",""2511"",""2516"",""2519"",""2522"",""2525"",""2529"",""2531"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""0.5666563946707227"",""3.2173086790405527"",""-0.7672747298808162"",""0.9733045171646266"",""1.473769079310224"",""-0.2914600533812482"",""-0.21261913890554268"",""-0.5032737201897166"",""0.29861046346916253""]}",0.0,NO
"{""type"":""0"",""size"":""2543"",""indices"":[""1"",""8"",""11"",""22"",""33"",""58"",""76"",""95"",""246"",""924"",""1677"",""2428"",""2433"",""2438"",""2443"",""2448"",""2453"",""2458"",""2463"",""2466"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2506"",""2511"",""2516"",""2519"",""2522"",""2525"",""2528"",""2532"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""-0.6879828146861925"",""-0.467650410164546"",""0.4525869203257326"",""-0.7853938799691554"",""-0.3718018596359379"",""-0.2914600533812482"",""-0.21261913890554268"",""1.0804289780222678"",""0.8157804701984116""]}",0.0,NO
"{""type"":""0"",""size"":""2543"",""indices"":[""1"",""7"",""11"",""24"",""31"",""63"",""76"",""95"",""219"",""929"",""1638"",""2428"",""2433"",""2438"",""2443"",""2448"",""2453"",""2458"",""2463"",""2466"",""2471"",""2476"",""2479"",""2484"",""2489"",""2494"",""2499"",""2502"",""2507"",""2511"",""2516"",""2519"",""2522"",""2525"",""2528"",""2531"",""2534"",""2535"",""2536"",""2537"",""2538"",""2539"",""2540"",""2541"",""2542""],""values"":[""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""1.0"",""-0.6879828146861925"",""-1.1376429718382004"",""-0.462309317329179"",""-0.1991610809245614"",""-1.2330682978108136"",""-0.2914600533812482"",""-0.21261913890554268"",""-0.5032737201897166"",""-1.7700695634478336""]}",1.0,>30
"{""type"":""0"",""size"":""2543"",""indices"":[""0"",""8"",""11"",""24"",""31"",""63"",""76"",""95"",""414"",""897"",""1970"",""2428"",""243


4. Class distribution:


label,readmitted,count,percentage
0.0,NO,54864,53.91
1.0,>30,35545,34.93
2.0,<30,11357,11.16



Text summary:
  label 0.0  (NO   )  count:  54,864  (53.91%)
  label 1.0  (>30  )  count:  35,545  (34.93%)
  label 2.0  (<30  )  count:  11,357  (11.16%)

=== VERIFICATION SUMMARY ===
  Rows preserved: 101,766 (expected 101,766) ✓
  Columns: ['features', 'label', 'readmitted']
  Three-class target preserved: ✓

Gold table 'healthcare_readmission.readmission.gold_patient_features' is ready for model training.
